# Lyden 100M — real-data training (M10) on Kaggle

Config: `configs/100m-kaggle.yaml` — vocab 32,768 · dim 512 · 16 layers (mmma) · d_state 128 · seq 1024 · batch 16 × grad_accum 2 = 32,768 tok/step · 7,600 steps ≈ 249M tokens · AMP fp16 (T4 tensor cores) · per-block activation checkpointing.

Run policy: **Save Version → Save & Run All (Commit)** with GPU T4x2 (or P100) and Internet ON.

- Evolution: pip → clone repo (persisted in /kaggle/working) → prepare TinyStories 200M shards (skipped if present) → train with auto-resume → eval.
- Sessions die at 12 h; checkpoints every 200 steps land in `/kaggle/working/Lyden/runs/100m/ckpts` and auto-resume on the next run.
- If fp16 is unstable or b16 OOMs, fall back inside cell 6: `--config configs/100m-kaggle-fallback.yaml` (batch 8 × accum 4) — same tokens/step and schedule.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import sys
print(sys.version)

In [ ]:
%pip install -q "mamba-ssm==2.3.2.post1" "causal-conv1d==1.7.0"
import torch
from mamba_ssm.modules.mamba2 import Mamba2
blk = Mamba2(d_model=512, d_state=128, d_conv=4, expand=2, layer_idx=0, use_mem_eff_path=False).cuda()
y = blk(torch.randn(2, 256, 512, device='cuda', dtype=torch.float16))
print('mamba-ssm OK:', y.shape, y.dtype, torch.__version__)

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /kaggle/working/Lyden/.git ]; then
  git clone https://github.com/mrrchandra/Lyden.git /kaggle/working/Lyden
else
  cd /kaggle/working/Lyden && git pull --ff-only || echo '(offline keep)'
fi
cd /kaggle/working/Lyden
python - <<'PYEOF'
import sys
sys.path.insert(0, 'src')
from lyden.config import load_config
c = load_config('configs/100m-kaggle.yaml')
t = c.training
print('config ok:', t.seq_len * t.batch_size * t.grad_accum, 'tok/step;', t.total_steps, 'steps;', t.amp)
PYEOF

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Lyden
if [ -f /kaggle/working/lyden-data/shards/train/manifest.json ]; then
  echo '[data] present, skipping preparation'
else
  python scripts/prepare_data.py \
      --dataset tinystories \
      --data-dir /kaggle/working/lyden-data \
      --tokenizer-sample 30000 \
      --max-train-tokens 200000000
fi
du -sh /kaggle/working/lyden-data

In [ ]:
%%bash
# Training with auto-resume; reruns resume from the latest ckpt under runs/100m/ckpts.
# If b16 OOMs or fp16 is unstable, switch --config to configs/100m-kaggle-fallback.yaml.
set -euo pipefail
cd /kaggle/working/Lyden
export PYTHONUNBUFFERED=1
for round in 1 2 3 4 5 6 7 8; do
  echo "[train round $round] start"
  if python scripts/train.py --config configs/100m-kaggle.yaml --val-every 600; then
    echo "[train round $round] completed"
    break
  else
    echo "[train round $round] exited nonzero; retrying from last checkpoint"
    sleep 5
  fi
done
ls -la /kaggle/working/Lyden/runs/100m/ckpts | tail -5

In [ ]:
%%bash
set -uo pipefail
cd /kaggle/working/Lyden
CKPT=$(ls -1 runs/100m/ckpts/ckpt_*.pt | sort | tail -1)
echo "[eval] $CKPT"
python scripts/eval.py \
    --config configs/100m-kaggle.yaml \
    --ckpt "$CKPT" \
    --tasks ppl,longctx,samples \
    --val-batches 40 \
    --lengths 512,1024 \
    --max-samples 3
echo '=== eval report ==='
cat runs/100m/eval_report.json | head -80